# 04 — Distinct ground-truth Yamada preservation benchmark

This notebook performs **one validation experiment only**.

For each independently generated ground-truth embedded graph \(G_{\rm true}^{(i)}\),

\[
G_{\rm true}^{(i)}
\;\xrightarrow{\text{Yamada}}\;
Y_{\rm true}^{(i)}
\]

and independently

\[
G_{\rm true}^{(i)}
\;\longrightarrow\;
\text{regular-neighborhood volume}
\;\longrightarrow\;
\text{skeletonization}
\;\longrightarrow\;
G_{\rm recovered}^{(i)}
\;\xrightarrow{\text{Yamada}}\;
Y_{\rm recovered}^{(i)}.
\]

The pass criterion for sample \(i\) is exactly

\[
\boxed{Y_{\rm true}^{(i)}=Y_{\rm recovered}^{(i)}}.
\]

The default benchmark generates **250 pairwise-distinct ground-truth graph topologies** in the subcubic/trivalent regime

\[
\Delta(G)\le 3.
\]

The number of ground-truth samples is controlled by one parameter: `N_GROUND_TRUTH`.

### Distinctness

Each ground truth is a connected, bridgeless cycle-plus-noncrossing-matching graph. A candidate is accepted only if its Weisfeiler–Lehman graph hash has not occurred before. Isomorphic graphs necessarily have the same WL hash, so accepting only new hashes prevents two isomorphic candidates from being counted as distinct. A hash collision can only reject an additional valid graph; it cannot admit an isomorphic duplicate.

The graph is embedded in 3D as a smooth lift \(z=f(x,y)\) of a crossing-free planar embedding. Its voxelized regular neighborhood is then passed through the production skeletonization/extraction functions.

> Matching Yamada polynomials shows preservation of the spatial-topological information detected by the normalized Yamada invariant on this benchmark. Yamada equality is not, in general, a complete proof of ambient isotopy.

In [1]:
# -----------------------------
# Adjustable benchmark settings
# -----------------------------

N_GROUND_TRUTH = 250       # <-- change this to 50, 100, 500, ...

RANDOM_SEED = 20260821

# Ground-truth graph generation
MIN_NODES = 7
MAX_NODES = 18
MAX_ALLOWED_DEGREE = 3
MIN_CHORD_SPAN = 3
MIN_VERTEX_ANGLE_DEG = 25.0
MIN_NONINCIDENT_CLEARANCE = 0.10

# Embedded geometry / voxelization
EMBEDDING_RADIUS = 0.82
EDGE_SAMPLES = 120
BOUND = 1.35
GRID_SIZE = 200
TUBE_RADIUS_VOX = 1

# Yamada computation
YAMADA_ROTATION = (0.0, 0.0, 0.0)
YAMADA_N_JOBS = 1

# Output
SAVE_RESULTS_CSV = True
RESULTS_CSV_NAME = "synthetic_ground_truth_yamada_preservation.csv"

assert N_GROUND_TRUTH > 0
assert MIN_NODES >= 4
assert MAX_NODES >= MIN_NODES
assert MAX_ALLOWED_DEGREE == 3
assert GRID_SIZE >= 32
assert TUBE_RADIUS_VOX >= 1

In [2]:
from __future__ import annotations

from pathlib import Path
import random
import time
import warnings

import networkx as nx
import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import display
from skimage.morphology import ball, dilation

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph repository checkout.")

import knotted_graph
from knotted_graph.core import remove_leaf_nodes, simplify_edges, smooth_edges
from knotted_graph.extraction import skeleton_image_to_graph, skeletonize_volume
from knotted_graph.invariants.yamada.native import native_available, native_import_error
from knotted_graph.projection import compute_yamada_polynomial

A = sp.Symbol("A")
DX = 2 * BOUND / (GRID_SIZE - 1)

print("Repository root:", ROOT)
print("KnottedGraph:", Path(knotted_graph.__file__).resolve())
print("Native Yamada backend:", native_available())
print("Native import error:", native_import_error())
assert native_available(), native_import_error()

Repository root: /Users/hakanakgun/Desktop/Projects/ProfLeeProjects/Knotted_graph_code_paper/LatestGit
KnottedGraph: /Users/hakanakgun/Desktop/Projects/ProfLeeProjects/Knotted_graph_code_paper/LatestGit/src/knotted_graph/__init__.py
Native Yamada backend: True
Native import error: None


In [3]:
def _cyclic_distance(i: int, j: int, n: int) -> int:
    d = abs(j - i)
    return min(d, n - d)


def _chords_cross_on_circle(a: int, b: int, c: int, d: int) -> bool:
    """Return True when two chords with distinct endpoints cross in the disk."""
    a, b = sorted((a, b))
    c, d = sorted((c, d))
    return (a < c < b < d) or (c < a < d < b)


def _random_noncrossing_matching(
    n: int,
    rng: random.Random,
    *,
    keep_probability: float,
) -> list[tuple[int, int]]:
    """
    Greedily generate a noncrossing matching of non-adjacent cycle vertices.

    Each vertex is used by at most one chord, so adding the matching to C_n
    keeps every vertex degree <= 3.
    """
    used: set[int] = set()
    chords: list[tuple[int, int]] = []

    candidates = [
        (i, j)
        for i in range(n)
        for j in range(i + 1, n)
        if _cyclic_distance(i, j, n) >= MIN_CHORD_SPAN
    ]
    rng.shuffle(candidates)

    for i, j in candidates:
        if i in used or j in used:
            continue
        if any(_chords_cross_on_circle(i, j, a, b) for a, b in chords):
            continue
        if rng.random() > keep_probability:
            continue

        chords.append((i, j))
        used.add(i)
        used.add(j)

    return sorted(chords)


def _circle_xy(n: int) -> np.ndarray:
    theta = 2 * np.pi * np.arange(n, dtype=float) / n
    return np.c_[
        EMBEDDING_RADIUS * np.cos(theta),
        EMBEDDING_RADIUS * np.sin(theta),
    ]


def _minimum_vertex_angle_deg(graph: nx.Graph, xy: np.ndarray) -> float:
    best = 180.0
    for v in graph.nodes:
        neighbors = list(graph.neighbors(v))
        vectors = []

        for u in neighbors:
            vector = xy[u] - xy[v]
            norm = float(np.linalg.norm(vector))
            if norm <= 1e-14:
                return 0.0
            vectors.append(vector / norm)

        for i in range(len(vectors)):
            for j in range(i + 1, len(vectors)):
                cosine = float(np.clip(vectors[i] @ vectors[j], -1.0, 1.0))
                angle = float(np.degrees(np.arccos(cosine)))
                best = min(best, angle)

    return best


def _segment_distance(p1, q1, p2, q2) -> float:
    """Euclidean distance between two closed 2D or 3D line segments."""
    p1 = np.asarray(p1, dtype=float)
    q1 = np.asarray(q1, dtype=float)
    p2 = np.asarray(p2, dtype=float)
    q2 = np.asarray(q2, dtype=float)

    u = q1 - p1
    v = q2 - p2
    w = p1 - p2

    a = float(u @ u)
    b = float(u @ v)
    c = float(v @ v)
    d = float(u @ w)
    e = float(v @ w)
    determinant = a * c - b * b

    if determinant < 1e-14:
        s = 0.0
        t = np.clip(e / c if c > 1e-14 else 0.0, 0.0, 1.0)
    else:
        s = np.clip((b * e - c * d) / determinant, 0.0, 1.0)
        t = np.clip((a * e - b * d) / determinant, 0.0, 1.0)

    if a > 1e-14:
        s = np.clip((b * t - d) / a, 0.0, 1.0)
    if c > 1e-14:
        t = np.clip((b * s + e) / c, 0.0, 1.0)

    return float(np.linalg.norm(w + s * u - t * v))


def _minimum_nonincident_clearance(graph: nx.Graph, xy: np.ndarray) -> float:
    edges = list(graph.edges())
    best = np.inf

    for index, (u, v) in enumerate(edges):
        for a, b in edges[index + 1:]:
            if {u, v} & {a, b}:
                continue

            best = min(
                best,
                _segment_distance(xy[u], xy[v], xy[a], xy[b]),
            )

    return float(best)


def _lift_xy(points_xy: np.ndarray, phase: float) -> np.ndarray:
    """
    Smooth injective lift of the xy plane into 3D.

    z is a single-valued function of (x,y), so two distinct xy points never
    become the same 3D point.
    """
    points_xy = np.asarray(points_xy, dtype=float)
    x = points_xy[:, 0]
    y = points_xy[:, 1]
    z = 0.075 * np.sin(1.7 * x + phase) * np.cos(1.9 * y - 0.5 * phase)
    return np.c_[x, y, z]


def _embed_graph(graph: nx.Graph, sample_index: int) -> nx.MultiGraph:
    n = graph.number_of_nodes()
    xy = _circle_xy(n)
    phase = 0.37 * sample_index

    embedded = nx.MultiGraph()
    node_xyz = _lift_xy(xy, phase)

    for node in graph.nodes:
        embedded.add_node(node, pos=node_xyz[node].copy())

    for u, v in graph.edges():
        edge_xy = np.linspace(xy[u], xy[v], EDGE_SAMPLES)
        edge_xyz = _lift_xy(edge_xy, phase)
        edge_xyz[0] = node_xyz[u]
        edge_xyz[-1] = node_xyz[v]
        embedded.add_edge(u, v, pts=edge_xyz)

    return embedded


def generate_distinct_ground_truths(
    n_samples: int,
    *,
    seed: int,
) -> list[dict]:
    """
    Generate pairwise non-isomorphic connected bridgeless subcubic graphs.

    Isomorphic graphs must have the same WL hash. Therefore accepting only
    previously unseen (node count, WL hash) keys prevents an isomorphic
    duplicate from entering the benchmark.
    """
    rng = random.Random(seed)
    accepted: list[dict] = []
    seen_keys: set[tuple[int, str]] = set()

    max_attempts = max(50_000, 500 * n_samples)

    for attempt in range(1, max_attempts + 1):
        n = rng.randint(MIN_NODES, MAX_NODES)
        keep_probability = rng.uniform(0.35, 0.95)

        chords = _random_noncrossing_matching(
            n,
            rng,
            keep_probability=keep_probability,
        )

        if not chords:
            continue

        graph = nx.cycle_graph(n)
        graph.add_edges_from(chords)

        if not nx.is_connected(graph):
            continue
        if list(nx.bridges(graph)):
            continue

        max_degree = max(dict(graph.degree()).values())
        if max_degree > MAX_ALLOWED_DEGREE:
            continue

        xy = _circle_xy(n)

        min_angle = _minimum_vertex_angle_deg(graph, xy)
        if min_angle < MIN_VERTEX_ANGLE_DEG:
            continue

        clearance = _minimum_nonincident_clearance(graph, xy)
        if clearance < MIN_NONINCIDENT_CLEARANCE:
            continue

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            wl_hash = nx.weisfeiler_lehman_graph_hash(graph)

        key = (n, wl_hash)
        if key in seen_keys:
            continue

        seen_keys.add(key)

        sample_id = len(accepted) + 1
        embedded = _embed_graph(graph, sample_id)

        accepted.append(
            {
                "id": sample_id,
                "graph": embedded,
                "abstract_graph": graph,
                "wl_hash": wl_hash,
                "n_nodes": graph.number_of_nodes(),
                "n_edges": graph.number_of_edges(),
                "max_degree": max_degree,
                "n_chords": len(chords),
                "min_vertex_angle_deg": min_angle,
                "min_nonincident_clearance": clearance,
            }
        )

        if len(accepted) == n_samples:
            break

    if len(accepted) != n_samples:
        raise RuntimeError(
            f"Generated only {len(accepted)}/{n_samples} distinct admissible "
            f"ground truths after {max_attempts} attempts. "
            "Increase MAX_NODES or relax only geometric admissibility thresholds; "
            "do not pad the benchmark with duplicates."
        )

    keys = [(row["n_nodes"], row["wl_hash"]) for row in accepted]

    assert len(keys) == len(set(keys)) == n_samples
    assert all(row["max_degree"] <= MAX_ALLOWED_DEGREE for row in accepted)
    assert all(nx.is_connected(row["abstract_graph"]) for row in accepted)
    assert all(not list(nx.bridges(row["abstract_graph"])) for row in accepted)

    return accepted

In [4]:
ground_truths = generate_distinct_ground_truths(
    N_GROUND_TRUTH,
    seed=RANDOM_SEED,
)

catalogue = pd.DataFrame(
    [
        {
            "id": row["id"],
            "nodes": row["n_nodes"],
            "edges": row["n_edges"],
            "max_degree": row["max_degree"],
            "chords": row["n_chords"],
            "min_vertex_angle_deg": row["min_vertex_angle_deg"],
            "min_nonincident_clearance": row["min_nonincident_clearance"],
            "wl_hash": row["wl_hash"],
        }
        for row in ground_truths
    ]
)

print(f"Generated {len(ground_truths)} distinct ground-truth graphs.")
print(
    "Unique (node-count, WL-hash) keys:",
    catalogue[["nodes", "wl_hash"]].drop_duplicates().shape[0],
)
print(
    "Node-count range:",
    int(catalogue["nodes"].min()),
    "to",
    int(catalogue["nodes"].max()),
)
print("Maximum degree:", int(catalogue["max_degree"].max()))

display(catalogue.head(10))

assert len(ground_truths) == N_GROUND_TRUTH
assert catalogue[["nodes", "wl_hash"]].drop_duplicates().shape[0] == N_GROUND_TRUTH
assert int(catalogue["max_degree"].max()) <= MAX_ALLOWED_DEGREE

Generated 250 distinct ground-truth graphs.
Unique (node-count, WL-hash) keys: 250
Node-count range: 7 to 18
Maximum degree: 3


,id,nodes,edges,max_degree,chords,min_vertex_angle_deg,min_nonincident_clearance,wl_hash
0,1,8,10,3,2,45.000000,0.443781,33373138243db2154505cbc1b8206c2c
1,2,12,14,3,2,45.000000,0.300141,f3e06ef904ee0627c755b01903502d82
2,3,16,19,3,3,33.750000,0.177754,60ea112df4c385d7acaf6cc32c3c1913
3,4,9,11,3,2,40.000000,0.360548,7f8d14af88d217aeea33d5fcec0f00c2
4,5,8,9,3,1,67.500000,0.579828,c65ea3ba48a0534eae2e8b3d16a065d9
5,6,7,8,3,1,51.428571,0.556327,0c733482b31918778705ff0dc463d97a
6,7,11,14,3,3,32.727273,0.249798,7d9ee31a9430b87cac21d73e086f671e
7,8,11,13,3,2,32.727273,0.249798,a3d9d72a36ddef3fba5003408071612c
8,9,15,18,3,3,36.000000,0.200420,b78d7b5b43b2e5a2c7ad2a273ec8d3ac
9,10,15,18,3,3,36.000000,0.200420,3ed561643791ace1714e188ff26b55d0


In [5]:
def _resample_polyline(points: np.ndarray, step: float) -> np.ndarray:
    points = np.asarray(points, dtype=float)
    parts = []

    for p, q in zip(points[:-1], points[1:]):
        length = float(np.linalg.norm(q - p))
        count = max(2, int(np.ceil(length / step)) + 1)
        parts.append(np.linspace(p, q, count, endpoint=False))

    parts.append(points[-1:])
    return np.vstack(parts)


def regular_neighborhood_volume(graph: nx.MultiGraph) -> np.ndarray:
    """
    Voxelize the embedded graph and thicken it into a regular-neighborhood volume.

    The boundary of this volume is the synthetic handlebody surface.
    """
    volume = np.zeros((GRID_SIZE, GRID_SIZE, GRID_SIZE), dtype=bool)

    for _, _, _, data in graph.edges(keys=True, data=True):
        points = _resample_polyline(
            np.asarray(data["pts"], dtype=float),
            DX / 3.0,
        )

        if np.any(points < -BOUND) or np.any(points > BOUND):
            raise ValueError("Embedded ground truth leaves the voxelization box.")

        indices = np.rint(
            (points + BOUND) / (2 * BOUND) * (GRID_SIZE - 1)
        ).astype(int)
        indices = np.clip(indices, 0, GRID_SIZE - 1)

        volume[
            indices[:, 0],
            indices[:, 1],
            indices[:, 2],
        ] = True

    return dilation(volume, footprint=ball(TUBE_RADIUS_VOX))


def _graph_from_voxel_to_world(graph: nx.MultiGraph) -> nx.MultiGraph:
    graph = nx.MultiGraph(graph)
    origin = np.array([-BOUND, -BOUND, -BOUND], dtype=float)

    for _, data in graph.nodes(data=True):
        data["pos"] = origin + DX * np.asarray(data["pos"], dtype=float)

    for _, _, _, data in graph.edges(keys=True, data=True):
        data["pts"] = origin + DX * np.asarray(data["pts"], dtype=float)

    return graph


def _cleanup_reconstructed_graph(graph: nx.MultiGraph) -> nx.MultiGraph:
    graph = remove_leaf_nodes(graph)
    graph = simplify_edges(graph)
    graph = smooth_edges(
        graph,
        epsilon=2 * DX,
        copy=False,
    )
    return graph


def reconstruct_from_regular_neighborhood(
    ground_truth: nx.MultiGraph,
) -> tuple[nx.MultiGraph, float, float]:
    volume = regular_neighborhood_volume(ground_truth)

    start = time.perf_counter()
    skeleton = skeletonize_volume(volume)
    skeleton_seconds = time.perf_counter() - start

    start = time.perf_counter()
    extracted = skeleton_image_to_graph(
        skeleton,
        max_junction_degree=MAX_ALLOWED_DEGREE,
    )
    extraction_seconds = time.perf_counter() - start

    recovered = _cleanup_reconstructed_graph(
        _graph_from_voxel_to_world(extracted)
    )

    return recovered, skeleton_seconds, extraction_seconds


def normalized_yamada(graph: nx.MultiGraph) -> sp.Expr:
    degrees = dict(graph.degree())
    max_degree = max(degrees.values(), default=0)

    if max_degree > MAX_ALLOWED_DEGREE:
        raise ValueError(
            f"Yamada benchmark requires max degree <= {MAX_ALLOWED_DEGREE}; "
            f"got {max_degree}."
        )

    value = compute_yamada_polynomial(
        graph,
        A,
        rotation_angles=YAMADA_ROTATION,
        normalize=True,
        n_jobs=YAMADA_N_JOBS,
    )
    return sp.expand(value)


def yamada_equal(first: sp.Expr, second: sp.Expr) -> bool:
    return sp.expand(first - second) == 0

## Run the single benchmark

For every distinct ground truth:

1. compute \(Y_{\rm true}\);
2. form its regular-neighborhood volume;
3. skeletonize and recover \(G_{\rm recovered}\);
4. compute \(Y_{\rm recovered}\);
5. test only whether \(Y_{\rm true}=Y_{\rm recovered}\).

There is no repeated-rotation benchmark and no separate special-case topology test.

In [6]:
records = []

for index, item in enumerate(ground_truths, start=1):
    graph_true = item["graph"]

    record = {
        "id": item["id"],
        "nodes_true": item["n_nodes"],
        "edges_true": item["n_edges"],
        "max_degree_true": item["max_degree"],
        "wl_hash": item["wl_hash"],
        "yamada_true": None,
        "nodes_recovered": None,
        "edges_recovered": None,
        "max_degree_recovered": None,
        "yamada_recovered": None,
        "yamada_match": False,
        "skeleton_seconds": None,
        "extraction_seconds": None,
        "error": None,
    }

    try:
        y_true = normalized_yamada(graph_true)

        graph_recovered, skeleton_seconds, extraction_seconds = (
            reconstruct_from_regular_neighborhood(graph_true)
        )

        recovered_degrees = dict(graph_recovered.degree())
        max_degree_recovered = max(recovered_degrees.values(), default=0)

        if max_degree_recovered > MAX_ALLOWED_DEGREE:
            raise RuntimeError(
                "Recovered graph left the admissible degree class: "
                f"max degree = {max_degree_recovered}."
            )

        y_recovered = normalized_yamada(graph_recovered)
        match = yamada_equal(y_true, y_recovered)

        record.update(
            {
                "yamada_true": str(y_true),
                "nodes_recovered": graph_recovered.number_of_nodes(),
                "edges_recovered": graph_recovered.number_of_edges(),
                "max_degree_recovered": max_degree_recovered,
                "yamada_recovered": str(y_recovered),
                "yamada_match": bool(match),
                "skeleton_seconds": skeleton_seconds,
                "extraction_seconds": extraction_seconds,
            }
        )

    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"

    records.append(record)

    if index == 1 or index % 10 == 0 or index == N_GROUND_TRUTH:
        passed_so_far = sum(row["yamada_match"] for row in records)
        print(
            f"[{index:>3}/{N_GROUND_TRUTH}] "
            f"Yamada matches so far: {passed_so_far}/{index}"
        )

results = pd.DataFrame(records)
display(results)

[  1/250] Yamada matches so far: 1/1
[ 10/250] Yamada matches so far: 10/10
[ 20/250] Yamada matches so far: 20/20
[ 30/250] Yamada matches so far: 30/30
[ 40/250] Yamada matches so far: 40/40
[ 50/250] Yamada matches so far: 50/50
[ 60/250] Yamada matches so far: 60/60
[ 70/250] Yamada matches so far: 70/70
[ 80/250] Yamada matches so far: 80/80
[ 90/250] Yamada matches so far: 90/90
[100/250] Yamada matches so far: 100/100
[110/250] Yamada matches so far: 110/110
[120/250] Yamada matches so far: 120/120
[130/250] Yamada matches so far: 130/130
[140/250] Yamada matches so far: 140/140
[150/250] Yamada matches so far: 150/150
[160/250] Yamada matches so far: 160/160
[170/250] Yamada matches so far: 170/170
[180/250] Yamada matches so far: 180/180
[190/250] Yamada matches so far: 190/190
[200/250] Yamada matches so far: 200/200
[210/250] Yamada matches so far: 210/210
[220/250] Yamada matches so far: 220/220
[230/250] Yamada matches so far: 230/230
[240/250] Yamada matches so far: 240/2

,id,nodes_true,edges_true,max_degree_true,wl_hash,yamada_true,nodes_recovered,edges_recovered,max_degree_recovered,yamada_recovered,yamada_match,skeleton_seconds,extraction_seconds,error
0,1,8,10,3,33373138243db2154505cbc1b8206c2c,-A**6 - A**5 - 3*A**4 - 2*A**3 - 3*A**2 - A - 1,4,6,3,-A**6 - A**5 - 3*A**4 - 2*A**3 - 3*A**2 - A - 1,True,0.020182,0.004822,None
1,2,12,14,3,f3e06ef904ee0627c755b01903502d82,-A**6 - A**5 - 3*A**4 - 2*A**3 - 3*A**2 - A - 1,4,6,3,-A**6 - A**5 - 3*A**4 - 2*A**3 - 3*A**2 - A - 1,True,0.022504,0.003206,None
2,3,16,19,3,60ea112df4c385d7acaf6cc32c3c1913,-A**8 - A**7 - 4*A**6 - 3*A**5 - 6*A**4 - 3*A*...,6,9,3,-A**8 - A**7 - 4*A**6 - 3*A**5 - 6*A**4 - 3*A*...,True,0.021342,0.003615,None
3,4,9,11,3,7f8d14af88d217aeea33d5fcec0f00c2,-A**6 - A**5 - 3*A**4 - 2*A**3 - 3*A**2 - A - 1,4,6,3,-A**6 - A**5 - 3*A**4 - 2*A**3 - 3*A**2 - A - 1,True,0.021514,0.003702,None
4,5,8,9,3,c65ea3ba48a0534eae2e8b3d16a065d9,-A**4 - A**3 - 2*A**2 - A - 1,2,3,3,-A**4 - A**3 - 2*A**2 - A - 1,True,0.020892,0.002822,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,246,16,19,3,a19b122125a524caa33add49493d2a2a,-A**8 - A**7 - 4*A**6 - 3*A**5 - 6*A**4 - 3*A*...,6,9,3,-A**8 - A**7 - 4*A**6 - 3*A**5 - 6*A**4 - 3*A*...,True,0.021099,0.003570,None
246,247,18,22,3,021779d733a976f066523e3c1acf485e,-A**10 - A**9 - 5*A**8 - 4*A**7 - 10*A**6 - 6*...,8,12,3,-A**10 - A**9 - 5*A**8 - 4*A**7 - 10*A**6 - 6*...,True,0.021157,0.006550,None
247,248,18,21,3,776dd58d70608c73205516c77905cc5e,-A**8 - A**7 - 4*A**6 - 3*A**5 - 6*A**4 - 3*A*...,6,9,3,-A**8 - A**7 - 4*A**6 - 3*A**5 - 6*A**4 - 3*A*...,True,0.020622,0.004073,None
248,249,18,22,3,37a8efc73dc61a9418d9129cb010ac7e,-A**10 - A**9 - 5*A**8 - 4*A**7 - 10*A**6 - 6*...,8,12,3,-A**10 - A**9 - 5*A**8 - 4*A**7 - 10*A**6 - 6*...,True,0.020490,0.006860,None


In [8]:
n_total = len(results)
n_pass = int(results["yamada_match"].sum())
n_fail = n_total - n_pass

print("=" * 72)
print("DISTINCT GROUND-TRUTH YAMADA PRESERVATION")
print("=" * 72)
print(f"Ground truths generated : {N_GROUND_TRUTH}")
print(f"Ground truths evaluated : {n_total}")
print(f"Yamada matches          : {n_pass}/{n_total}")
print(f"Yamada failures         : {n_fail}/{n_total}")
print(f"Accuracy                : {100.0 * n_pass / n_total:.2f}%")

assert n_total == N_GROUND_TRUTH

failures = results.loc[
    ~results["yamada_match"],
    [
        "id",
        "nodes_true",
        "edges_true",
        "max_degree_true",
        "nodes_recovered",
        "edges_recovered",
        "max_degree_recovered",
        "yamada_true",
        "yamada_recovered",
        "error",
    ],
]

if len(failures):
    print("\nFailures:")
    display(failures)
else:
    print(
        "\nPASS: every distinct ground-truth graph preserved the normalized "
        "Yamada polynomial through regular-neighborhood construction, "
        "skeletonization, and graph recovery."
    )

if SAVE_RESULTS_CSV:
    csv_path = ROOT / "dev" / RESULTS_CSV_NAME
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    results.to_csv(csv_path, index=False)
    print("\nSaved:", csv_path)

# The single scientific acceptance criterion of this notebook.
assert n_fail == 0, (
    f"Benchmark failed: {n_fail}/{n_total} reconstructed graphs did not preserve "
    "the normalized Yamada polynomial. Inspect the displayed failure table."
)

DISTINCT GROUND-TRUTH YAMADA PRESERVATION
Ground truths generated : 250
Ground truths evaluated : 250
Yamada matches          : 250/250
Yamada failures         : 0/250
Accuracy                : 100.00%

PASS: every distinct ground-truth graph preserved the normalized Yamada polynomial through regular-neighborhood construction, skeletonization, and graph recovery.

Saved: /Users/hakanakgun/Desktop/Projects/ProfLeeProjects/Knotted_graph_code_paper/LatestGit/dev/synthetic_ground_truth_yamada_preservation.csv


## Interpretation

If the final assertion passes at the default setting,

\[
\boxed{250/250}
\]

distinct ground truths satisfy

\[
Y_{\rm true}=Y_{\rm recovered}.
\]

The empirical statement is:

> Across 250 pairwise-distinct connected bridgeless subcubic ground-truth graphs, the regular-neighborhood \(\rightarrow\) skeletonization \(\rightarrow\) recovered-graph pipeline preserved the normalized Yamada invariant in every tested case.

This is the only scientific acceptance criterion in this notebook.